# Lab 4 — Seaborn Visualizations (Zomato EDA)

**Day 02 · Python for Data Science**

## Goals

1. Build histograms, box plots, violin plots, and heatmaps with Seaborn.
2. Combine matplotlib subplots into dashboards.
3. Save figures for reports; optional Plotly for interactivity.

> **Quick check:** `rating_distribution.png` · mean rating ≈ **3.70** · top avg-cost city **Kolkata**




**Day 2 flow:** Labs 1–3 structures/NumPy/pandas → **Lab 4 (you are here)** visualization → Labs 5–6 regression.

## Why visualize before modeling?

Never fit a model on data you have not plotted.

---

## 1. Setup

In [ ]:
%matplotlib inline

from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from IPython.display import Image, display

GH_ROOT = Path.cwd().resolve()
if GH_ROOT.name == "notebooks":
    GH_ROOT = GH_ROOT.parents[2]
elif (GH_ROOT.parent / "notebooks").is_dir() and (GH_ROOT.parents[1] / "requirements-student.txt").is_file():
    GH_ROOT = GH_ROOT.parents[1]
else:
    for parent in [GH_ROOT, *GH_ROOT.parents]:
        if (parent / "requirements-student.txt").is_file():
            GH_ROOT = parent
            break

ZOMATO_CSV = GH_ROOT / "data" / "zomato" / "zomato_restaurants.csv"
OUTPUT_DIR = GH_ROOT / "hands-on" / "02-python-for-data-science" / "output"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
sns.set_theme(style="whitegrid")

print("Output:", OUTPUT_DIR)

In [ ]:
df = pd.read_csv(ZOMATO_CSV)
print(f"Loaded {df.shape[0]} rows")
assert df.shape[0] == 500

---

## 2. Rating distribution

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
sns.histplot(df["aggregate_rating"], bins=15, kde=True, ax=ax, color="steelblue")
ax.set_title("Distribution of aggregate rating")
ax.set_xlabel("aggregate_rating")
plt.tight_layout()
plt.show()


### 2b. KDE-only overlay

In [ ]:
fig, ax = plt.subplots(figsize=(6, 3))
sns.kdeplot(df["aggregate_rating"], fill=True, ax=ax, color="steelblue", alpha=0.4)
ax.set_title("Rating density")
plt.tight_layout()
plt.show()


---

## 3. Cost by city — box plot (top 5 cities)

In [ ]:
top_cities = df["city"].value_counts().head(5).index.tolist()
city_subset = df[df["city"].isin(top_cities)]
print("Top 5 cities:", top_cities)

fig, ax = plt.subplots(figsize=(7, 4))
sns.boxplot(data=city_subset, x="city", y="average_cost_for_two", ax=ax, palette="pastel")
ax.set_title("Cost for two — top 5 cities")
ax.tick_params(axis="x", rotation=30)
plt.tight_layout()
plt.show()


### 3b. Violin plot — same cities

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
sns.violinplot(data=city_subset, x="city", y="average_cost_for_two", ax=ax, palette="muted")
ax.set_title("Cost distribution (violin)")
ax.tick_params(axis="x", rotation=30)
plt.tight_layout()
plt.show()


### 3c. Bar chart — mean rating by city (top 8)

In [ ]:
city_rating = df.groupby("city")["aggregate_rating"].mean().sort_values(ascending=False).head(8)
fig, ax = plt.subplots(figsize=(7, 4))
city_rating.plot(kind="bar", ax=ax, color="coral")
ax.set_title("Mean rating by city (top 8)")
ax.set_ylabel("aggregate_rating")
plt.tight_layout()
plt.show()


---

## 4. Scatter — rating vs cost

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
sns.scatterplot(data=df, x="average_cost_for_two", y="aggregate_rating",
                hue="online_order", alpha=0.5, ax=ax)
ax.set_title("Rating vs cost")
plt.tight_layout()
scatter_plot = OUTPUT_DIR / "rating_vs_cost_scatter.png"
fig.savefig(scatter_plot, dpi=100)
plt.show()
print("saved:", scatter_plot.name)


### 4b. Joint plot (sample 120 rows)

In [ ]:
sample = df.sample(120, random_state=42)
g = sns.jointplot(data=sample, x="votes", y="aggregate_rating", kind="scatter", height=4)
g.figure.suptitle("Votes vs rating (sample)", y=1.02)
plt.show()


---

## 5. Correlation heatmap

In [ ]:
num = df[["aggregate_rating", "votes", "average_cost_for_two"]]
fig, ax = plt.subplots(figsize=(4, 3))
sns.heatmap(num.corr(), annot=True, fmt=".2f", cmap="coolwarm", ax=ax)
ax.set_title("Numeric correlations")
plt.tight_layout()
plt.show()


---

## 6. Categorical counts

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 3))
sns.countplot(data=df, x="online_order", ax=axes[0], palette="Set2")
axes[0].set_title("Online order")
sns.countplot(data=df, x="book_table", ax=axes[1], palette="Set3")
axes[1].set_title("Book table")
plt.tight_layout()
plt.show()


### 6b. Strip plot — ratings by online order

In [ ]:
fig, ax = plt.subplots(figsize=(5, 4))
sns.stripplot(data=df.sample(200, random_state=1), x="online_order", y="aggregate_rating",
              alpha=0.4, ax=ax)
ax.set_title("Rating spread by delivery flag")
plt.tight_layout()
plt.show()


---

## 7. Dashboard + save checkpoint plot

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
sns.histplot(df["aggregate_rating"], bins=15, kde=True, ax=axes[0], color="steelblue")
axes[0].set_title("Distribution of aggregate rating")
sns.boxplot(data=city_subset, x="city", y="average_cost_for_two", ax=axes[1], palette="pastel")
axes[1].set_title("Cost for two — top 5 cities")
axes[1].tick_params(axis="x", rotation=30)
fig.tight_layout()
rating_plot = OUTPUT_DIR / "rating_distribution.png"
fig.savefig(rating_plot, dpi=100, bbox_inches="tight")
plt.show()
print(f"Saved: {rating_plot}")
assert rating_plot.is_file()
display(Image(filename=str(rating_plot)))


### 7b. Facet grid — rating vs cost by city (sample)

In [ ]:
g = sns.FacetGrid(df.sample(150, random_state=7), col="city", col_wrap=3, sharex=True, sharey=True, height=2.2)
g.map_dataframe(sns.scatterplot, x="average_cost_for_two", y="aggregate_rating", alpha=0.5)
g.figure.suptitle("Rating vs cost — sample cities", y=1.02)
plt.show()


### 7c. Pairplot on numeric columns

In [ ]:
cols = ["aggregate_rating", "votes", "average_cost_for_two"]
sns.pairplot(df[cols].sample(100, random_state=3), corner=True, diag_kind="hist", height=2)
plt.suptitle("Numeric pairplot (sample)", y=1.02)
plt.show()


---

## 8. Plotly (interactive — course topic)




In [ ]:
try:
    import plotly.express as px
    fig = px.scatter(df, x="average_cost_for_two", y="aggregate_rating",
                     color="online_order", hover_data=["city", "votes"],
                     title="Zomato — interactive rating vs cost")
    fig.show()
    html_out = OUTPUT_DIR / "rating_vs_cost_plotly.html"
    fig.write_html(html_out)
    print("saved:", html_out.name)
except ImportError:
    print("Plotly not installed — optional")


---

## 9. Numeric checkpoints

In [ ]:
mean_rating = df["aggregate_rating"].mean()
city_means = df.groupby("city")["average_cost_for_two"].mean().sort_values(ascending=False)
top_city = city_means.index[0]
print(f"mean rating: {mean_rating:.2f}")
print(f"top city by avg cost: {top_city} ({city_means.iloc[0]:.0f})")
assert abs(mean_rating - 3.70) < 0.05
assert top_city == "Kolkata"
print("\n✓ Checkpoint assertions passed")


**Previous:** [Lab 3](lab03_pandas_zomato_load.ipynb)  
**Next:** [Lab 5](lab05_linear_regression_fit.ipynb)